# ЛАБОРАТОРНА РОБОТА  1

In [4]:
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import FuncAnimation, PillowWriter

## Генетичний алгоритм: 
### $$ x = \frac{\mathrm{Dec}(\mathrm{code})}{2^{w} - 1} \cdot (b - a) + a $$

In [5]:
w = 16
pop_size = 60
generations = 35
p_crossover = 0.85
p_mutate = 0.01

In [3]:
def decode(chromosome, bounds):
    
    dim = len(bounds)
    values = []

    for i in range(dim):
        
        a, b = bounds[i]
        
        part = chromosome[ (i * w) : ((i + 1) * w) ]
        dec = int(part, 2)     
        
        x = dec / (2**w - 1) * (b - a) + a
        
        values.append(x)

    return values

def init_population(dim):
    
    population = [] 
    total_bits = w * dim  
  
    for i in range(pop_size):
        chromosome = "" 
        for j in range(total_bits):
            
            bit = random.choice('01') 
            chromosome += bit

        population.append(chromosome)
    
    return population  
    

def selection(population, fitness_values, minimize=True):
    
    i, j = random.sample(range(len(population)), 2)

    if minimize:
        return population[i] if fitness_values[i] < fitness_values[j] else population[j]
    else:
        return population[i] if fitness_values[i] > fitness_values[j] else population[j]

def crossover(p1, p2, dim):

    if random.random() < p_crossover:
        point = random.randint(1, (w * dim) - 1)
        c1 = p1[:point] + p2[point:]
        c2 = p2[:point] + p1[point:]
        return c1, c2
    
    return p1, p2

def mutate(chromosome):
    
    new_chromosome = ""
    
    for bit in chromosome:
        if random.random() < p_mutate:
            new_chromosome += '1' if bit == '0' else '0'
        else:
            new_chromosome += bit
            
    return new_chromosome

def ga(fit_func, bounds, minimize=True):

    dim = len(bounds)
    population = init_population(dim)
    
    history_best = []
    population_history = []

    for i in range(generations):

        decoded_population = []
        for chromosome in population:
            value = decode(chromosome, bounds)  
            decoded_population.append(value)  

        fitness_values = []  
        for individual in decoded_population:
            fitness = fit_func(individual)  
            fitness_values.append(fitness)  

        if minimize:
            best_index = np.argmin(fitness_values)
        else:
            best_index = np.argmax(fitness_values)

        history_best.append(fitness_values[best_index])
        population_history.append(decoded_population)

        best_chromosome = population[best_index]
        new_population = [best_chromosome]

        while len(new_population) < pop_size:

            p1 = selection(population, fitness_values, minimize)
            p2 = selection(population, fitness_values, minimize)

            c1, c2 = crossover(p1, p2, dim)
            c1 = mutate(c1)
            c2 = mutate(c2)

            new_population.extend([c1, c2])

        population = new_population[:pop_size]

    best_chromosome = population[best_index]
    
    best_x_vector = decode(best_chromosome, bounds)
    best_value = fitness_values[best_index]

    return best_x_vector, best_value, history_best, population_history

## Алгоритм зграї сірих вовків
### $$ x_i^{new} = x_i + step \cdot \frac{GBest - x_i}{\|GBest - x_i\|} $$

In [6]:
def gwo(func, a, b, dim, step, s = pop_size, minimize=True):

    wolves = np.random.uniform(a, b, (s, dim))
    
    fitness_value = []
    for wolf in wolves:
        value = func(wolf)          
        fitness_value.append(value)  
    fitness_value = np.array(fitness_value)

    if minimize:
        index_best = np.argmin(fitness_value)  
    else:
        index_best = np.argmax(fitness_value)
    
    GBest = wolves[index_best].copy()
    GBest_value = fitness_value[index_best]

    history = []
    population_history = []

    for k in range(generations):
        
        for i in range(s):
            
            if i == index_best:
                continue

            vec = GBest - wolves[i]
            norm = np.linalg.norm(vec)
            wolves[i] = wolves[i] + step * vec / norm
            
            wolves[i] = np.clip(wolves[i], a, b)

        fitness_value = []
        for wolf in wolves:
            value = func(wolf)          
            fitness_value.append(value)  
        fitness_value = np.array(fitness_value)
        
        if minimize:
            index_best = np.argmin(fitness_value)  
        else:
            index_best = np.argmax(fitness_value)

        if (minimize and fitness_value[index_best] < GBest_value) or \
           (not minimize and fitness_value[index_best] > GBest_value):

            GBest = wolves[index_best].copy()
            GBest_value = fitness_value[index_best]

        population_history.append(wolves.copy())
        history.append(GBest_value)

    return GBest, GBest_value, history, population_history

## 1. Тестування алгоритмів на одновимірній тестовій одноекстремальній функції. Процес пошуку глобального екстремуму.

In [1]:
def create_graph_gif(population_history, history, f, bounds, filename, minimize=True):
    
    a, b = bounds
    x_plot = np.linspace(a, b, 200)
    y_plot = f(x_plot)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))  

    ax1 = axes[0]
    ax1.plot(x_plot, y_plot, color="blue")
    scat = ax1.scatter([], [], s=40, color="red") 
    best_point = ax1.scatter([], [], s=120, marker='o', color="green")  
    ax1.set_xlim(a, b)
    ax1.set_ylim(min(y_plot) - 1, max(y_plot) + 1)
    ax1.set_title("Графік функції")
    ax1.grid()

    ax2 = axes[1]
    ax2.set_xlim(0, len(history))
    buffer = 0.05 * (max(history) - min(history))
    ax2.set_ylim(min(history) - buffer, max(history) + buffer)
    line, = ax2.plot([], [], color="green", lw=2)
    ax2.set_title("Графік збіжності")
    ax2.set_xlabel("Ітерація")
    ax2.set_ylabel("Найкраще f(x)")
    ax2.grid()

    def update(frame):
        x_pop = np.array(population_history[frame])
        y_pop = f(x_pop)

        scat.set_offsets(np.c_[x_pop, y_pop])
        if minimize:
            best_idx = np.argmin(y_pop)
        else:
            best_idx = np.argmax(y_pop)
        best_point.set_offsets([[x_pop[best_idx], y_pop[best_idx]]])

        line.set_data(np.arange(frame+1), history[:frame+1])

        return scat, best_point, line

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(population_history),
        interval=300,
        blit=True,
        repeat=False
    )

    ani.save(filename, writer="pillow")
    plt.close(fig)

### 1.1 Гармонійна функція: $$ f(x) = x^3 (3 - x)^5 \sin(10\pi x), \quad x \in [0,3], \quad f \to \min $$

In [2]:
def harmonic_func(x):
    return x**3 * (3 - x)**5 * np.sin(10 * np.pi * x)

a1, b1 = 0, 3

In [12]:
best_x_ga, best_val_ga, hist_ga, pop_history_h_ga = ga(lambda x: harmonic_func(x[0]), [(a1, b1)], minimize=True)
best_x_gwo, best_val_gwo, hist_gwo, pop_history_h_gwo = gwo(harmonic_func, a1, b1, dim=1, step=0.02, minimize=True )

print("Генетичний алгоритм: ")
print(f"x = {best_x_ga[0]:.10f}, f(x) = {best_val_ga:.10f}")
print("Алгоритм зграї сірих вовків: ")
print(f"x = {best_x_gwo[0]:.10f}, f(x) = {float(best_val_gwo):.10f}")

Генетичний алгоритм: 
x = 1.1499198901, f(x) = -32.9574842428
Алгоритм зграї сірих вовків: 
x = 1.1510293370, f(x) = -32.9368579073


C:\Users\BOS\AppData\Local\Temp\ipykernel_2316\2134903710.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print(f"x = {best_x_gwo[0]:.10f}, f(x) = {float(best_val_gwo):.10f}")


In [13]:
create_graph_gif(
    population_history = pop_history_h_ga,
    history = hist_ga,         
    f = harmonic_func,
    bounds = (a1, b1),
    filename = "one_dim_func/harmonic_ga.gif",
    minimize = True
)

create_graph_gif(
    population_history = pop_history_h_gwo,
    history = hist_gwo,         
    f = harmonic_func,
    bounds = (a1, b1),
    filename = "one_dim_func/harmonic_gwo.gif",
    minimize = True
)

### 1.2 Параметрична функція: $$ f(x) = (a - x)^m (b - x)^n \sin(p\pi x)\sin(q\pi x), \quad x \in [a,b], \quad f \to \max $$
### $$ a,b \in \mathbb{R}, \quad  m,n \in \mathbb{Z}^+, \quad  p > 5, \quad q > 7, \quad  p,q \in \mathbb{R} $$

In [46]:
a2 = 0
b2 = 4
m = 2
n = 3
p = 6.5
q = 8.3

def parametric_func(x):
    return (a2 - x)**m * (b2 - x)**n * np.sin(p * np.pi * x) * np.sin(q * np.pi * x)

In [47]:
best_x_ga, best_val_ga, hist_ga, pop_history_p_ga = ga(lambda x: parametric_func(x[0]), [(a2, b2)], minimize=False)
best_x_gwo, best_val_gwo, hist_gwo, pop_history_p_gwo = gwo(parametric_func ,a2, b2, dim=1, step=0.03,minimize=False)

print("Генетичний алгоритм: ")
print(f"x = {best_x_ga[0]:.10f}, f(x) = {best_val_ga:.10f}")
print("Алгоритм зграї сірих вовків: ")
print(f"x = {best_x_gwo[0]:.10f}, f(x) = {float(best_val_gwo):.10f}")

Генетичний алгоритм: 
x = 1.1486991684, f(x) = 30.2431156809
Алгоритм зграї сірих вовків: 
x = 1.1481919698, f(x) = 30.2383269193


C:\Users\BOS\AppData\Local\Temp\ipykernel_18544\1102025636.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print(f"x = {best_x_gwo[0]:.10f}, f(x) = {float(best_val_gwo):.10f}")


In [48]:
create_graph_gif(
    population_history = pop_history_h_ga,
    history = hist_ga,         
    f = parametric_func,
    bounds = (a2, b2),
    filename = "one_dim_func/parametric_ga.gif",
    minimize = False
)

create_graph_gif(
    population_history = pop_history_h_gwo,
    history = hist_gwo,         
    f = parametric_func,
    bounds = (a2, b2),
    filename = "one_dim_func/parametric_gwo.gif",
    minimize = False
)

## 2. Пошук глобального екстремуму двовимірної функції.

In [14]:
def create_all_graph_gif( population_history, best_history,f, x_bounds, y_bounds, filename):

    x = np.linspace(x_bounds[0], x_bounds[1], 100)
    y = np.linspace(y_bounds[0], y_bounds[1], 100)
    X, Y = np.meshgrid(x, y)
    Z = f((X, Y))

    fig = plt.figure(figsize=(12, 8))

    ax3d = fig.add_subplot(2, 2, 1, projection='3d')
    ax_contour = fig.add_subplot(2, 2, 2)
    ax_hist = fig.add_subplot(2, 2, 3)
    ax_conv = fig.add_subplot(2, 2, 4)

    def update(frame):

        ax3d.clear()
        ax_contour.clear()
        ax_hist.clear()
        ax_conv.clear()

        pop = population_history[frame]

        pop_x = np.array([p[0] for p in pop])
        pop_y = np.array([p[1] for p in pop])
        pop_z = np.array([f([px, py]) for px, py in zip(pop_x, pop_y)])

        ax3d.plot_surface(X, Y, Z, alpha=0.6)
        ax3d.scatter(pop_x, pop_y, pop_z, color='red')
        ax3d.set_title(f"3D Графік (Покоління {frame})")
        ax3d.set_xlim(x_bounds)
        ax3d.set_ylim(y_bounds)
        
        ax_contour.contourf(X, Y, Z, levels=50)
        ax_contour.scatter(pop_x, pop_y, color='red')
        ax_contour.set_title("Контурний графік")
        ax_contour.set_xlim(x_bounds)
        ax_contour.set_ylim(y_bounds)

        fitness = pop_z
        ax_hist.hist(fitness, bins=15)
        ax_hist.set_title("Графік пристосованості")

        ax_conv.plot(best_history)
        ax_conv.grid(True)
        ax_conv.set_title("Графік збіжності")
        ax_conv.set_xlabel("Ітерація")
        ax_conv.set_ylabel("Найкраще значення")

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(population_history),
        interval = 300,
        repeat=False
    )

    ani.save(filename, writer="pillow")
    plt.close(fig)

In [15]:
def f_isoma(v):
    x, y = v
    return -np.cos(x) * np.cos(y) * np.exp(-(x - np.pi)**2 - (y - np.pi)**2)

def f_rastrigin(v, A=10):
    x = np.array(v)
    n = len(x)
    return A*n + np.sum(x**2 - A*np.cos(2*np.pi*x))

def f_ackley(v):
    x, y = v
    return (-20*np.exp(-0.2*np.sqrt(0.5*(x**2 + y**2)))
            - np.exp(0.5*(np.cos(2*np.pi*x) + np.cos(2*np.pi*y)))
            + np.e + 20)

def f_cross_in_tray(v):
    x, y = v
    term = np.abs(np.sin(x) * np.sin(y) *
                  np.exp(np.abs(100 - np.sqrt(x**2 + y**2)/np.pi)))
    return -0.0001 * (term + 1)**0.1
    
def f_egg_holder(v):
    x, y = v
    return (-(y + 47)*np.sin(np.sqrt(np.abs(x/2 + y + 47)))
            - x*np.sin(np.sqrt(np.abs(x - y - 47))))   
    
def f_holder(v):
    x, y = v
    return -np.abs(np.sin(x) * np.cos(y) *
                   np.exp(np.abs(1 - np.sqrt(x**2 + y**2)/np.pi)))
    
def f_schaffer_1(v):
    x, y = v
    numerator = np.sin(x**2 - y**2)**2 - 0.5
    denominator = (1 + 0.001*(x**2 + y**2))**2
    return 0.5 + numerator/denominator

def f_schaffer_2(v):
    x, y = v
    numerator = np.cos(np.sin(np.abs(x**2 - y**2)))**2 - 0.5
    denominator = (1 + 0.001*(x**2 + y**2))**2
    return 0.5 + numerator/denominator

In [16]:
FUNCTIONS = {

    "isoma": {
        "func": f_isoma,
        "bounds": [(0, 2*np.pi), (0, 2*np.pi)],
        "name": "Isoma function"
    },

    "rastrigin": {
        "func": f_rastrigin,
        "bounds": [(-5.12, 5.12), (-5.12, 5.12)],
        "name": "Rastrigin function"
    },

    "ackley": {
        "func": f_ackley,
        "bounds": [(-5, 5), (-5, 5)],
        "name": "Ackley function"
    },

    "cross_in_tray": {
        "func": f_cross_in_tray,
        "bounds": [(-10, 10), (-10, 10)],
        "name": "Cross-in-Tray function"
    },

    "egg_holder": {
        "func": f_egg_holder,
        "bounds": [(-512, 512), (-512, 512)],
        "name": "Egg Holder function"
    },

    "holder": {
        "func": f_holder,
        "bounds": [(-10, 10), (-10, 10)],
        "name": "Holder function"
    },

    "schaffer1": {
        "func": f_schaffer_1,
        "bounds": [(-100, 100), (-100, 100)],
        "name": "Schaffer 1 function"
    },

    "schaffer2": {
        "func": f_schaffer_2,
        "bounds": [(-100, 100), (-100, 100)],
        "name": "Schaffer 2 function"
    }
}

def get_function(key):

    key = key.lower()

    func_data = FUNCTIONS[key]
    f = func_data["func"]
    bounds = func_data["bounds"]
    a = (bounds[0][0], bounds[0][1])
    b = (bounds[1][0], bounds[1][1])
    name = func_data["name"]
    
    return f, a, b, name

In [ ]:
for key in FUNCTIONS.keys():

    f, a, b, name = get_function(key) 
    best_x_ga, best_val_ga, hist_ga, pop_history_ga = ga(f, (a, b), minimize=True)
    filename = f"two_dim_func/{key}_ga.gif"
    
    create_all_graph_gif(pop_history_ga, hist_ga, f, a, b, filename)
    
for key in FUNCTIONS.keys():
    
    f, a, b, name = get_function(key) 
    best_x_gwo, best_val_gwo, hist_gwo, pop_history_gwo = gwo(f, a, b,  dim=2, step=0.02, minimize=True) 
    filename = f"two_dim_func/{key}_gwo.gif"
    
    create_all_graph_gif(pop_history_ga, hist_gwo, f, a, b, filename)

C:\Users\BOS\AppData\Local\Temp\ipykernel_18544\3212181260.py:31: RuntimeWarning: invalid value encountered in divide
  wolves[i] = wolves[i] + step * vec / norm
C:\Users\BOS\AppData\Local\Temp\ipykernel_18544\3212181260.py:31: RuntimeWarning: invalid value encountered in divide
  wolves[i] = wolves[i] + step * vec / norm
C:\Users\BOS\AppData\Local\Temp\ipykernel_18544\3212181260.py:31: RuntimeWarning: invalid value encountered in divide
  wolves[i] = wolves[i] + step * vec / norm
C:\Users\BOS\AppData\Local\Temp\ipykernel_18544\3212181260.py:31: RuntimeWarning: invalid value encountered in divide
  wolves[i] = wolves[i] + step * vec / norm
C:\Users\BOS\AppData\Local\Temp\ipykernel_18544\3212181260.py:31: RuntimeWarning: invalid value encountered in divide
  wolves[i] = wolves[i] + step * vec / norm
C:\Users\BOS\AppData\Local\Temp\ipykernel_18544\3212181260.py:31: RuntimeWarning: invalid value encountered in divide
  wolves[i] = wolves[i] + step * vec / norm
C:\Users\BOS\AppData\Local\T

### 3. Для багатовимірних одноекстремальних функцій показати лише графік пристосованості популяції, а також показати графік відстані від кращого елемента популяції до відомого оптимального значення функції, яке вказане.

### Функція Растринга:
$$ f(X) = A n + \sum_{i=1}^{n} \left(x_i^2 - A \cos(2 \pi x_i)\right), \quad x_i \in [-5.12, 5.12], \quad f \to \min $$

In [ ]:
def create_multidim_gif(population_history, f, filename, min_val):

    best_dist_history = []

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax_hist, ax_conv = axes

    def update(frame):
        ax_hist.cla()
        ax_conv.cla()

        pop = population_history[frame]  
        fitness = np.array([f(ind) for ind in pop])
        
        ax_hist.hist(fitness, bins=15)
        ax_hist.set_title("Графік пристосованості")

        best_idx = np.argmin(fitness)
        best_val = fitness[best_idx]
        dist_to_opt = best_val - min_val
        best_dist_history.append(dist_to_opt)

        ax_conv.plot(best_dist_history, color="green", lw=2)
        ax_conv.set_title("Відстань до глобального мінімуму")
        ax_conv.set_xlabel("Ітерація")
        ax_conv.set_ylabel("|f(best) - f_opt|")
        ax_conv.set_xlim(0, len(population_history))
        buffer = 0.05 * max(best_dist_history) if best_dist_history else 1
        ax_conv.set_ylim(0, max(best_dist_history) + buffer)
        ax_conv.grid(True)

        return ax_hist, ax_conv

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(population_history),
        interval=400,
        repeat=False
    )

    ani.save(filename, writer="pillow")
    plt.close(fig)

In [65]:
n_dim = 4
bounds_rastrigin = [(-5.12, 5.12)] * n_dim
a = np.array([bound[0] for bound in bounds_rastrigin])
b = np.array([bound[1] for bound in bounds_rastrigin])

In [96]:
best_point_ga, best_value_ga, history_ga, pop_history_ga = ga(f_rastrigin, bounds_rastrigin, minimize=True)
print(best_value_ga)

0.11335128008597195


In [ ]:
create_multidim_gif(pop_history_ga, f_rastrigin, "multi_dim_func/rastrigin_ga.gif", min_val = 0)

In [131]:
best_point_gwo, best_value_gwo, history_gwo, pop_history_gwo = gwo(f_rastrigin, a, b,  dim=n_dim, step=0.02, minimize=True)
print(best_value_gwo)

15.138423251808426


In [ ]:
create_multidim_gif(pop_history_gwo, f_rastrigin, "multi_dim_func/rastrigin_gwo.gif", min_val = 0)